# Tab-pretext split v2 — colours vs morphology
The hard-group model (`tab-mdn-hard-v1`, 6 features) already lifted u−g R² 0.59→0.83.
This splits the hard group once more, one model per sub-group:

- **COLOR**: `u-g`, `i-z`, `r-i` — the z-sensitive colours
- **MORPH**: `conc_r`, `log_petroR90_r` — concentration / outer radius

Question: does another round of dedication buy more R², or has that lever saturated?
Reference columns: the joint 16-feature model AND the 6-feature hard model.

Runtime → Change runtime type → **GPU**.


In [ ]:
import os
os.environ['CUTOUT_SIZE'] = '24'        # sample_v4.5 is 24x24
![ -d /content/Photo-Z-CNN-Model ] || git clone -b main https://github.com/zhhrozhh/Photo-Z-CNN-Model.git /content/Photo-Z-CNN-Model
%cd /content/Photo-Z-CNN-Model
!pip -q install mlflow


In [ ]:
# Google auth — if 'credential propagation was unsuccessful', just re-run this cell
from google.colab import auth; auth.authenticate_user()
!gcloud config set project macrocosm-lewagon -q


In [ ]:
!mkdir -p /content/data
!gcloud storage cp gs://macrocosm-lewagon/data/sample_v4.5/catalog_v4.parquet /content/data/catalog_v1.parquet
!gcloud storage cp gs://macrocosm-lewagon/data/sample_v4.5/images_*.npy /content/data/
DATA_DIR = '/content/data'

import tensorflow as tf
print('GPUs =', tf.config.list_physical_devices('GPU'))


In [ ]:
try:
    from google.colab import userdata
    MLFLOW_TOKEN = userdata.get('MLFLOW_TOKEN')
except Exception as e:
    print('no MLFLOW_TOKEN secret ->', e); MLFLOW_TOKEN = None


## Groups + reference R² (v4-5 val)


In [ ]:
COLOR = ['u-g', 'i-z', 'r-i']
MORPH = ['conc_r', 'log_petroR90_r']

REF = {  # feature: (joint 16-feat model, dedicated 6-feat hard model)
    'u-g':            (.588, .826),
    'i-z':            (.728, .835),
    'r-i':            (.861, .932),
    'conc_r':         (.713, .787),
    'log_petroR90_r': (.828, .842)}


## Train the two models


In [ ]:
from tab_cnn import train_tab

hist_c, model_color, (fm_c, fs_c) = train_tab(
    data_dir=DATA_DIR, crop=24, preproc='p99', features=COLOR,
    N=None, batch=256, lr=3e-4, epochs=50,
    run_name='tab-mdn-color-v1', mlflow_token=MLFLOW_TOKEN)


In [ ]:
hist_m, model_morph, (fm_m, fs_m) = train_tab(
    data_dir=DATA_DIR, crop=24, preproc='p99', features=MORPH,
    N=None, batch=256, lr=3e-4, epochs=50,
    run_name='tab-mdn-morph-v1', mlflow_token=MLFLOW_TOKEN)


## Compare per-feature R² — joint vs hard(6) vs dedicated sub-group


In [ ]:
import glob, re
import numpy as np, pandas as pd
import eval as ev, fusion as fu
from tab_cnn import tab_targets

cat, z_all, o2i, Y = tab_targets(DATA_DIR)
va = np.array([o2i[int(o)] for o in pd.read_csv('splits/v4-5-validate.csv').objid if int(o) in o2i])

SHARD = 6000
mm = {int(re.findall(r'images_(\d+)_', p)[0]) // SHARD: np.load(p, mmap_mode='r')
      for p in glob.glob(f'{DATA_DIR}/images_*.npy')}
X = np.empty((len(va), 24, 24, 5), np.float16)
for s in np.unique(va // SHARD):
    sel = np.where(va // SHARD == s)[0]; rr = va[sel] % SHARD; srt = np.argsort(rr)
    X[sel[srt]] = mm[int(s)][rr[srt]]
pp = ev.make_np_preprocess('p99')
Xp = pp(np.asarray(X, 'float32'))

def per_feature_r2(model, feats, fm, fs):
    raw = model.predict(Xp, batch_size=1024, verbose=0)
    pi, mu = raw[..., :2], raw[..., 2:4]
    pred = (pi * mu).sum(-1) * fs + fm
    out = {}
    for j, name in enumerate(feats):
        i = fu.TAB_FEATURES.index(name)
        m = np.isfinite(Y[va, i])
        resid = pred[m, j] - Y[va, i][m]
        out[name] = 1 - np.var(resid) / np.var(Y[va, i][m])
    return out

r2 = {**per_feature_r2(model_color, COLOR, fm_c, fs_c),
      **per_feature_r2(model_morph, MORPH, fm_m, fs_m)}

print(f"{'feature':<16} {'joint16':>8} {'hard6':>8} {'dedicated':>10} {'d_vs_hard6':>11}")
for name, v in r2.items():
    j, h = REF[name]
    print(f'{name:<16} {j:8.3f} {h:8.3f} {v:10.3f} {v - h:+11.3f}')


## Save embeddings for HGB stacking at home


In [ ]:
import tensorflow as tf
for tag, model in [('color', model_color), ('morph', model_morph)]:
    embn = tf.keras.Model(model.input, model.get_layer('embedding').output)
    paths = sorted(glob.glob(f'{DATA_DIR}/images_*.npy'), key=lambda p: int(re.findall(r'images_(\d+)_', p)[0]))
    emb = np.concatenate([embn.predict(pp(np.asarray(np.load(p), 'float32')), batch_size=1024, verbose=0)
                          for p in paths])
    np.save(f'/content/emb_tabmdn_{tag}_v1.npy', emb)
    print(tag, emb.shape)
!gcloud storage cp /content/emb_tabmdn_color_v1.npy /content/emb_tabmdn_morph_v1.npy gs://macrocosm-lewagon/results/
